In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)
import pandas as pd
import matplotlib.pyplot as plt
from time import time
from pathlib import Path
from tqdm import tqdm

import matplotlib as mpl

from tysserand import tysserand as ty
from mosna import mosna

import matplotlib as mpl
mpl.rcParams["figure.facecolor"] = 'white'
mpl.rcParams["axes.facecolor"] = 'white'
mpl.rcParams["savefig.facecolor"] = 'white'
plt.style.use('seaborn-v0_8-poster')

In [2]:
RUN_LONG = True
data_dir_codex = Path("../data/CODEX_CTCL")
data_dir_IMC = Path("../data/MBTMEIMCPublic_IMC_Breast_Cancer")

In [3]:
trim_dist = False # or 'percentile_size'
min_neighbors = 3 # number of nearest nodes to connect for each unconneted node
n_shuffle = 500

time_assort = []

In [4]:
import numpy as np

def canon_edges(df):
    """Arêtes non orientées, dédupliquées : (min, max) pour chaque paire."""
    s = df["source"].to_numpy()
    t = df["target"].to_numpy()
    # np.minimum / np.maximum marchent aussi sur des chaînes
    out = pd.DataFrame({"source": np.minimum(s, t), "target": np.maximum(s, t)})
    return out.drop_duplicates()

## CODEX

### Load data

In [5]:
try:
    objects_path = data_dir_codex / "41467_2021_26974_MOESM3_ESM_-_Objects.parquet"
    obj = pd.read_parquet(data_dir_codex / "41467_2021_26974_MOESM3_ESM_-_Objects.parquet")
except FileNotFoundError:
    objects_path = data_dir_codex / "41467_2021_26974_MOESM3_ESM_-_Objects.xlsx"
    obj = pd.read_excel(objects_path, skiprows=2)
    # for latter use
    obj.to_parquet(data_dir_codex / "41467_2021_26974_MOESM3_ESM_-_Objects.parquet")
obj

,Count,EventCount,Index in File,ClusterID,FileName,ClusterName,Spots,Patients,Groups,FOXP3,...,PD-1+CD8+,CD4+Tcells,PD-1+tumor,CD8+Tcells,Ki67+CD4+,Ki-67+Tregs,ICOS+Tregs,IDO+Tregs,Ki67+tumor,original_index
0,0,1,60,33987,reg001,B cells,1,1,1,0.692403,...,0,0,0,0,0,0,0,0,0,0
1,1,1,94,33987,reg001,B cells,1,1,1,8.094747,...,0,0,0,0,0,0,0,0,0,1
2,2,1,198,33987,reg001,B cells,1,1,1,4.110389,...,0,0,0,0,0,0,0,0,0,2
3,3,1,284,33987,reg001,B cells,1,1,1,0.000000,...,0,0,0,0,0,0,0,0,0,3
4,4,1,298,33987,reg001,B cells,1,1,1,2.937673,...,0,0,0,0,0,0,0,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117165,116917,1,633,33981,reg066,vasculature,66,14,4,84.269714,...,0,0,0,0,0,0,0,0,0,116917
117166,116918,1,677,33981,reg066,vasculature,66,14,4,56.425945,...,0,0,0,0,0,0,0,0,0,116918
117167,116919,1,681,33981,reg066,vasculature,66,14,4,63.152092,...,0,0,0,0,0,0,0,0,0,116919
117168,116920,1,705,33981,reg066,vasculature,66,14,4,66.873306,...,0,0,0,0,0,0,0,0,0,116920


### Prepare data

In [6]:
# sample_cols = obj.columns[:10]
obj.rename(columns={'X': 'x', 'Y': 'y', 'Z':'z', 'ClusterName': 'cell type'}, inplace=True)
patient_col = 'Patients'
sample_col = 'FileName'
group_col = 'Groups'
# slices are 5µm thick, so data is pseudo-2D, including z leads to spatial networks reconstruction artifacts
pos_cols = ['x', 'y']
pheno_col = 'cell type'  # unique columns with all cell types
sample_cols = [sample_col, pheno_col, patient_col, 'Spots' ,group_col]
marker_cols = obj.columns[9:67]
marker_cols = [x for x in marker_cols if x != 'HOECHST1:Cyc_1_ch_1']
# pos_cols = obj.columns[69:76]
cell_type_cols = list(obj.columns[75:96]) # one-hot encoded cell types
marker_posit_cols = list(obj.columns[100:-1])
all_cols = sample_cols + marker_cols + pos_cols + cell_type_cols + marker_posit_cols
net_cols = pos_cols + [pheno_col] + cell_type_cols + marker_cols
nb_clusters = obj[pheno_col].unique().size

In [7]:
with pd.option_context('display.max_rows', 40):
    print(obj[pheno_col].value_counts())

cell type
tumor cells                     38499
epithelium                      18182
macrophages (M1>M2)             13800
stroma                          11210
Tregs                            7733
vasculature                      5404
CD8+ T cells                     5350
macrophages (M1=M2)              4426
B cells                          2205
macrophages (M2>M1)              1859
Langerhans cells                 1744
CD4+ T cells                     1647
tumor cells, intraepithelial     1316
lymphatics                        783
plasma cells                      725
mast cells                        619
nerves                            572
neutrophils                       430
IDO+ stromal cells                237
CLA+ leukocytes                   218
DCs, CD11c+                       211
Name: count, dtype: int64


In [8]:
uniq_patients = obj[patient_col].unique()
uniq_samples = obj[sample_col].unique()
n_uniq_patients = len(uniq_patients)
n_uniq_samples = len(uniq_samples)

processed_dir = Path("../data/processed/CODEX_CTCL")
dir_fig_save = processed_dir / 'figures'
net_size_threshold = 0

reconst_dir = processed_dir / f"tysserand_python_network"
save_dir = reconst_dir / "figures"
save_dir.mkdir(parents=True, exist_ok=True)
edges_dir = reconst_dir
nodes_dir = reconst_dir

### Do the Benchmark

In [9]:
from time import time
tab=[]
if RUN_LONG:
    for i, patient_id in tqdm(enumerate(uniq_patients), total=n_uniq_patients):
        select_patient = obj[patient_col] == patient_id
        samples = obj.loc[select_patient, sample_col].unique()
        for j, sample_id in enumerate(samples):
            select_sample = obj[sample_col] == sample_id
            nodes = obj.loc[select_patient & select_sample, net_cols]
            coords = nodes[pos_cols].values
            t_min = time()
            pairs = ty.build_delaunay(
                coords, 
                node_adaptive_trimming=True, 
                n_edges=3, 
                trim_dist_ratio=2,
                min_dist=0, 
                trim_dist=trim_dist,
                )
            t_final = time()
            tab.append([patient_id,sample_id,t_final-t_min, len(coords), "CODEX_CTCL"])

            if min_neighbors > 0:
                pairs = ty.link_solitaries(
                    coords, 
                    pairs, 
                    min_neighbors=min_neighbors,
                    verbose=0)

            edges = pd.DataFrame(data=pairs, columns=['source', 'target'])
            edges.to_parquet(edges_dir / f'edges_patient-{patient_id}_sample-{sample_id}.parquet', index=False)

            nodes.to_parquet(nodes_dir / f'nodes_patient-{patient_id}_sample-{sample_id}.parquet', index=False)

    df = pd.DataFrame(tab, columns=['patient','sample','time', 'number of cells', 'dataset'])
    df.to_csv(processed_dir / f'benchmark_times.csv', index=False, mode='a', sep=',')

 57%|█████▋    | 8/14 [00:04<00:03,  1.70it/s]

all nodes have at least 3 edges


 64%|██████▍   | 9/14 [00:05<00:04,  1.24it/s]

all nodes have at least 3 edges


100%|██████████| 14/14 [00:10<00:00,  1.32it/s]


In [10]:
!mosna tysserand-network --file ./config_CODEX.yaml --working_dir ../data/processed/CODEX_CTCL/MOSNA_GUI/ --benchmark

[QT_INFO] [INFO] Parameters are read correctly
[QT_PROGRESS] current=0 total=69 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=1 total=69 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=2 total=69 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=3 total=69 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=4 total=69 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=5 total=69 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=6 total=69 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=7 total=69 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=8 total=69 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=9 total=69 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=10 total=69 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=11 total=69 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=12 total=69 desc=[PROCESS] Verification of all file
[

In [11]:
print(f"Average time per sample: {sum([x[2] for x in tab]) / len(tab):.2f} seconds")
print(f"Total time per sample: {sum([x[2] for x in tab])} seconds")

Average time per sample: 0.12 seconds
Total time per sample: 8.051432609558105 seconds


### Comparaison Network Construction

In [12]:
edges_dir_old = edges_dir
edges_dir_new = Path('../data/processed/CODEX_CTCL/MOSNA_GUI/temp/net_dir_mosna')

In [13]:
results = []

for i, patient_id in tqdm(enumerate(uniq_patients), total=n_uniq_patients):
    select_patient = obj[patient_col] == patient_id
    samples = obj.loc[select_patient, sample_col].unique()

    for j, sample_id in enumerate(samples):
        fname = f'edges_patient-{patient_id}_sample-{sample_id}.parquet'
        edges_old = canon_edges(pd.read_parquet(edges_dir_old / fname))
        edges_new = canon_edges(pd.read_parquet(edges_dir_new / fname))

        df_old = edges_old[["source", "target"]].drop_duplicates()
        df_new = edges_new[["source", "target"]].drop_duplicates()

        merged = df_old.merge(df_new, on=["source", "target"], how="outer", indicator=True)

        n_communes = (merged["_merge"] == "both").sum()
        n_old_only = (merged["_merge"] == "left_only").sum()
        n_new_only = (merged["_merge"] == "right_only").sum()

        results.append({
            "dataset": "CODEX_CTCL",
            "patient": patient_id,
            "sample": sample_id,
            "n_old": len(df_old),
            "n_new": len(df_new),
            "communes": n_communes,
            "seulement_old": n_old_only,
            "seulement_new": n_new_only,
            "jaccard": n_communes / len(merged) if len(merged) else 1.0,
        })

df_results_CODEX = pd.DataFrame(results)

  0%|          | 0/14 [00:00<?, ?it/s]

100%|██████████| 14/14 [00:00<00:00, 33.55it/s]


### Assortativity

In [ ]:
if RUN_LONG:
    print("Computing z-scored assortativity")
    start = time()
    net_stats_CODEX = mosna.groups_assort_mixmat(
        net_dir=edges_dir, 
        attributes_col=pheno_col, 
        make_onehot=True,
        id_level_1='patient',
        id_level_2='sample', 
        extension='parquet',
        n_shuffle=n_shuffle,
        parallel_groups='max',  # or False
        save_intermediate_results=True)
    end = time()
    duration = end - start
    print(f"Assortativity took {duration:.3f}s for n_suffle {n_shuffle}")

    time_assort.append(['CODEX', duration])

    net_stats_CODEX.index = net_stats_CODEX['id']
    net_stats_CODEX.drop(columns=['id'], inplace=True)

2026-11-11 17:46:48,404 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 42GB due to system memory limit of 31.04 GiB
2026-11-11 17:46:48,406 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 42GB due to system memory limit of 31.04 GiB
2026-11-11 17:46:48,408 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 42GB due to system memory limit of 31.04 GiB
2026-11-11 17:46:48,410 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 42GB due to system memory limit of 31.04 GiB
2026-11-11 17:46:48,412 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 42GB due to system memory limit of 31.04 GiB
2026-11-11 17:46:48,413 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 42GB due to system memory limit of 31.04 GiB


2026-11-11 17:46:48,415 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 42GB due to system memory limit of 31.04 GiB
2026-11-11 17:46:48,418 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 42GB due to system memory limit of 31.04 GiB
2026-11-11 17:46:48,419 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 42GB due to system memory limit of 31.04 GiB
2026-11-11 17:46:48,420 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 42GB due to system memory limit of 31.04 GiB
2026-11-11 17:46:48,421 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 42GB due to system memory limit of 31.04 GiB
2026-11-11 17:46:48,422 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 42GB due to system memory limit of 31.04 GiB
2026-11-11 17:46:48,423 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 42GB due to system memory limit of 31.04 GiB
2026-11-11 17:46:48,425 - d

Computing z-scored assortativity


2026-11-11 17:47:02,066 - distributed.worker - ERROR - Compute Failed
Key:       make_group_network_stats-937f180c-0c7c-416c-b8c9-2085c1627a28
State:     executing
Task:  <Task 'make_group_network_stats-937f180c-0c7c-416c-b8c9-2085c1627a28' make_group_network_stats(, ...)>
Exception: 'ValueError("columns overlap but no suffix specified: Index([\'B cells\', \'CD4+ T cells\', \'CD8+ T cells\', \'CLA+ leukocytes\',\\n       \'DCs, CD11c+\', \'IDO+ stromal cells\', \'Langerhans cells\', \'Tregs\',\\n       \'epithelium\', \'lymphatics\', \'macrophages (M1=M2)\',\\n       \'macrophages (M1>M2)\', \'macrophages (M2>M1)\', \'mast cells\', \'nerves\',\\n       \'plasma cells\', \'stroma\', \'tumor cells\', \'tumor cells, intraepithelial\',\\n       \'vasculature\'],\\n      dtype=\'object\')")'
Traceback: '  File "/home/owen.griere/Desktop/MOSNA_GUI/mosna-package/mosna/mosna.py", line 1335, in make_group_network_stats\n    nodes = nodes.join(pd.get_dummies(nodes[attributes_col], prefix=\'\', p

In [ ]:
!mosna assortativity --file ./config_CODEX.yaml --working_dir ../data/processed/CODEX_CTCL/MOSNA_GUI/ --benchmark

## IMC Breast Cancer

### Load and prepare data

In [ ]:
objects_path = data_dir_IMC / "SingleCells.csv"

if objects_path.with_suffix('.parquet').exists():
    obj = pd.read_parquet(objects_path.with_suffix('.parquet'))
else:
    obj = pd.read_csv(objects_path)
    obj.to_parquet(objects_path.with_suffix('.parquet'))
obj

,ImageNumber,ObjectNumber,metabric_id,cellPhenotype,is_epithelial,is_tumour,is_normal,is_dcis,is_interface,is_perivascular,...,CD31-vWF,CXCL12,HLA-ABC,panCK,c-Caspase3,DNA1,DNA2,Location_Center_X,Location_Center_Y,AreaShape_Area
0,1,1,MB-0282,CK^{med}ER^{lo},1,1,0,0,1,0,...,0.495330,0.401104,9.282180,18.180412,0.163026,29.445302,54.989698,121.418605,4.023256,43
1,1,2,MB-0282,ER^{hi}CXCL12^{+},1,1,0,0,1,0,...,0.552035,2.567602,1.417755,48.593726,0.310149,23.160734,42.596685,150.468531,6.083916,143
2,1,4,MB-0282,CD4^{+} T cells & APCs,0,1,0,0,0,0,...,0.493669,0.117742,3.791699,3.404438,0.242591,16.884008,31.325295,221.930233,5.139535,129
3,1,5,MB-0282,CK^{med}ER^{lo},1,1,0,0,0,0,...,0.373286,1.048929,2.364418,23.289196,0.297700,8.475436,15.654673,354.136364,5.890909,110
4,1,7,MB-0282,CD4^{+} T cells,0,1,0,0,1,0,...,0.287512,0.058795,4.690886,0.664581,0.078051,14.009167,28.587826,469.174603,7.158730,126
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1123461,797,3818,MB-0406,CK8-18^{hi}CXCL12^{hi},1,1,0,0,0,0,...,0.412993,7.255870,0.947830,76.300971,0.000000,16.235925,31.912625,482.950000,947.825000,40
1123462,797,3819,MB-0406,CK8-18^{hi}CXCL12^{hi},1,1,0,0,0,0,...,0.609522,3.370633,1.534348,53.953806,0.205527,7.448000,12.347708,605.791667,945.333333,24
1123463,797,3820,MB-0406,CK8-18^{hi}CXCL12^{hi},1,1,0,0,1,0,...,0.562381,2.553417,0.629895,41.573201,0.187847,9.222450,18.821950,833.200000,947.350000,20
1123464,797,3822,MB-0406,CK^{lo}ER^{lo},1,1,0,0,1,0,...,0.133439,0.019060,1.156061,0.053584,0.000000,3.641944,7.286806,851.916667,949.222222,36


In [ ]:
obj.rename(columns={'Location_Center_X': 'x', 'Location_Center_Y': 'y'}, inplace=True)
sample_col = 'ImageNumber'  # != from `sample_cols`
patient_col = 'metabric_id'
pheno_col = 'cellPhenotype'
sample_cols = [sample_col, patient_col, pheno_col, 'ObjectNumber']
# this file contains 'c-Caspase3c-PARP' instead of 'c-Caspase3c' as in the dataframe
# all_epitopes = pd.read_csv(data_dir / 'markerStackOrder.csv').iloc[:, 1].values
# so instead we directly do:
all_epitopes = obj.columns[11:50].values
# remove Histone H3 and DNA markers
marker_cols = list(all_epitopes[1:-2])
pos_cols = ['x', 'y']
cell_type_cols = [
    'is_epithelial',
    'is_tumour',
    'is_normal',
    'is_dcis',
    'is_interface',
    'is_perivascular',
    'is_hotAggregate',
    ]
# make the phenotypes more readable
pheno_mapper = {x : x.replace('^{', '').
                      replace('}', '').
                      replace('_{', '_').
                      replace('hiC', 'hi C').
                      replace('hiE', 'hi E').
                      replace('loE', 'lo E').
                      replace('medE', 'med E') for x in obj[pheno_col].unique()}
obj[pheno_col] = obj[pheno_col].map(pheno_mapper)

nb_phenotypes = obj[pheno_col].unique().size
all_cols = sample_cols + marker_cols + pos_cols + cell_type_cols + [pheno_col]
# columns we want to include in network data
net_cols = pos_cols + [pheno_col] + cell_type_cols + marker_cols

print(f'nb phenotypes: {nb_phenotypes}')
print(f'nb used markers: {len(marker_cols)}')

nb phenotypes: 32
nb used markers: 36


In [ ]:
# Show number of cells per sample
sample_sizes = obj[['ImageNumber', 'ObjectNumber']].groupby(['ImageNumber']).count()
sample_sizes

,ObjectNumber
ImageNumber,
1,1624
2,1057
3,1648
4,1348
5,2018
...,...
793,372
794,4460
795,2689


In [ ]:
samples_dst = obj[sample_col].unique()[:10]
uniq_patients = obj['metabric_id'].unique()

### Benchmark

In [ ]:
output_dir = Path("../data/processed/IMC_Breast_Cancer")
edges_dir = Path("../data/processed/IMC_Breast_Cancer/tysserand_python_network")
edges_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
if RUN_LONG:
    for i, patient_id in tqdm(enumerate(uniq_patients), total=len(uniq_patients)):
        select_patient = obj[patient_col] == patient_id
        samples = obj.loc[select_patient, sample_col].unique()
        for j, sample in enumerate(samples):
            select_sample = obj[sample_col] == sample
            nodes = obj.loc[select_patient & select_sample, :]
            coords = nodes[pos_cols].values
            t_start = time()
            pairs = ty.build_delaunay(
                coords, 
                node_adaptive_trimming=True, 
                n_edges=3, 
                trim_dist_ratio=2,
                min_dist=0, 
                trim_dist=trim_dist,
                )
            t_final = time()
            tab.append([patient_id,sample,t_final-t_start, len(coords), "IMC Breast Cancer"])
            
            if min_neighbors > 0:
                pairs = ty.link_solitaries(
                    coords, 
                    pairs, 
                    min_neighbors=min_neighbors,
                    verbose=0,
                    )

            edges = pd.DataFrame(data=pairs, columns=['source', 'target'])
            edges.to_parquet(edges_dir / f'edges_patient-{patient_id}_sample-{sample}.parquet', index=False)
            nodes.to_parquet(edges_dir / f'nodes_patient-{patient_id}_sample-{sample}.parquet', index=False)

    df = pd.DataFrame(tab, columns=["patient","sample","time", "number of cells", "dataset"])
    df.to_csv(output_dir / f'benchmark_times.csv', index=False,)

  0%|          | 0/718 [00:00<?, ?it/s]

all nodes have at least 3 edges


  1%|          | 4/718 [00:00<01:18,  9.05it/s]

all nodes have at least 3 edges


  1%|          | 5/718 [00:00<01:49,  6.52it/s]

all nodes have at least 3 edges


... (251 outputs left out)

all nodes have at least 3 edges


 99%|█████████▉| 712/718 [01:47<00:00, 10.37it/s]

all nodes have at least 3 edges
all nodes have at least 3 edges


100%|██████████| 718/718 [01:51<00:00,  6.43it/s]


In [ ]:
!mosna tysserand-network --file ./config_IMC_Breast_Cancer.yaml --working_dir ../data/processed/IMC_Breast_Cancer/MOSNA_GUI --benchmark
print(f"Average time per sample: {sum([x[2] for x in tab]) / len(tab):.2f} seconds")
print(f"Total time per sample: {sum([x[2] for x in tab])} seconds")

[QT_INFO] [INFO] Parameters are read correctly
[QT_PROGRESS] current=0 total=794 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=1 total=794 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=2 total=794 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=3 total=794 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=4 total=794 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=5 total=794 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=6 total=794 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=7 total=794 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=8 total=794 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=9 total=794 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=10 total=794 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=11 total=794 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=12 total=794 desc=[PROCESS] Verification 

### Benchmark plotting

In [ ]:
edges_dir_old = edges_dir
edges_dir_new = Path('../data/processed/IMC_Breast_Cancer/MOSNA_GUI/temp/net_dir_mosna')

In [ ]:
results = []

for i, patient_id in tqdm(enumerate(uniq_patients), total=n_uniq_patients):
    samples = obj.loc[obj[patient_col] == patient_id, sample_col].unique()

    for sample_id in samples:
        fname = f'edges_patient-{patient_id}_sample-{sample_id}.parquet'
        cols = ["source", "target"]
        df_old = canon_edges(pd.read_parquet(edges_dir_old / fname, columns=cols))
        df_new = canon_edges(pd.read_parquet(edges_dir_new / fname, columns=cols))

        merged = df_old.merge(df_new, on=["source", "target"],
                              how="outer", indicator=True)

        n_communes = (merged["_merge"] == "both").sum()
        n_old_only = (merged["_merge"] == "left_only").sum()
        n_new_only = (merged["_merge"] == "right_only").sum()

        results.append({
            "dataset": "IMC Breast Cancer",
            "patient": patient_id,
            "sample": sample_id,
            "n_old": len(df_old),
            "n_new": len(df_new),
            "communes": n_communes,
            "seulement_old": n_old_only,
            "seulement_new": n_new_only,
            "jaccard": n_communes / len(merged) if len(merged) else 1.0,
        })

df_results_IMC = pd.DataFrame(results)

718it [00:27, 25.68it/s]                       


### Assortativity

In [ ]:
if RUN_LONG:
    print("Computing z-scored assortativity")
    start = time()
    net_stats_IMC = mosna.groups_assort_mixmat(
        net_dir=edges_dir, 
        attributes_col=pheno_col, 
        make_onehot=True,
        id_level_1='patient',
        id_level_2='sample', 
        extension='parquet',
        n_shuffle=n_shuffle,
        parallel_groups='max',  # or False
        save_intermediate_results=True)
    end = time()
    duration = end - start
    print(f"Assortativity took {duration:.3f}s for n_suffle {n_shuffle}")

    time_assort.append(['IMC Breast Cancer', duration])

    net_stats_IMC.index = net_stats_IMC['id']
    net_stats_IMC.drop(columns=['id'], inplace=True)

In [ ]:
!mosna assortativity --file ./config_IMC_Breast_Cancer.yaml --working_dir ../data/processed/IMC_Breast_Cancer/MOSNA_GUI --benchmark

## SYNTHETIC

### Benchmark

In [ ]:
output_dir = Path("../data/processed/SYNTHETIC")
edges_dir = Path("../data/processed/SYNTHETIC/tysserand_python_network")

nodes_dir = Path("../data/SYNTHETIC/")
edges_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
import multiprocessing as mp
from functools import partial
from time import perf_counter

import pandas as pd
from tqdm import tqdm


def process_file(file, pos_cols, trim_dist, min_neighbors, edges_dir):
    patient_id = file.stem.split("patient-")[-1]

    nodes = pd.read_parquet(file)
    try:
        coords = nodes[pos_cols].values
    except KeyError:
        coords = nodes[["X", "Y"]].values

    t_start = perf_counter()
    pairs = ty.build_delaunay(
        coords,
        node_adaptive_trimming=True,
        n_edges=3,
        trim_dist_ratio=2,
        min_dist=0,
        trim_dist=trim_dist,
    )
    elapsed = perf_counter() - t_start

    if min_neighbors > 0:
        pairs = ty.link_solitaries(
            coords,
            pairs,
            min_neighbors=min_neighbors,
            verbose=0,
        )

    edges = pd.DataFrame(data=pairs, columns=["source", "target"])
    edges.to_parquet(edges_dir / f"edges_patient-{patient_id}.parquet", index=False)
    nodes.to_parquet(edges_dir / f"nodes_patient-{patient_id}.parquet", index=False)

    return [patient_id, None, elapsed, len(coords), "SYNTHETIC"]


if __name__ == "__main__":  # required on Windows/macOS (spawn); remove in a notebook on Linux
    files = sorted(nodes_dir.glob("nodes*.parquet"))

    if RUN_LONG:
        worker = partial(
            process_file,
            pos_cols=[["X", "Y"]],
            trim_dist=trim_dist,
            min_neighbors=min_neighbors,
            edges_dir=edges_dir,
        )

        n_workers = max(1, mp.cpu_count() - 1)
        with mp.Pool(processes=n_workers) as pool:
            tab = list(tqdm(pool.imap(worker, files, chunksize=1), total=len(files)))

        df = pd.DataFrame(
            tab, columns=["patient", "sample", "time", "number of cells", "dataset"]
        )
        df.to_csv(output_dir / "benchmark_times.csv", index=False)

  0%|          | 0/200 [00:00<?, ?it/s]Exception ignored in: Process ForkPoolWorker-10:
Exception ignored in: Process ForkPoolWorker-1:
Process ForkPoolWorker-17:
  0%|          | 0/200 [00:02<?, ?it/s]<bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7b77e35af750>>
<bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7b77e35af750>>Traceback (most recent call last):
  File "/home/owen.griere/miniconda3/envs/mosna-GUI/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    
def _clean_thread_parent_frames(Traceback (most recent call last):

  File "/home/owen.griere/miniconda3/envs/mosna-GUI/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames

    def _clean_thread_parent_frames(

KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
!mosna tysserand-network --file ./config_SYN.yaml --working_dir ../data/processed/SYNTHETIC/MOSNA_GUI --benchmark

print(f"Average time per sample: {sum([x[2] for x in tab]) / len(tab):.2f} seconds")
print(f"Total time per sample: {sum([x[2] for x in tab])} seconds")

[QT_INFO] [INFO] Parameters are read correctly
[QT_PROGRESS] current=0 total=204 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=1 total=204 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=2 total=204 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=3 total=204 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=4 total=204 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=5 total=204 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=6 total=204 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=7 total=204 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=8 total=204 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=9 total=204 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=10 total=204 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=11 total=204 desc=[PROCESS] Verification of all file
[QT_PROGRESS] current=12 total=204 desc=[PROCESS] Verification 

### Benchmark Plotting

In [ ]:
edges_dir_old = edges_dir
edges_dir_new = Path('../data/processed/SYNTHETIC/MOSNA_GUI/temp/net_dir_mosna')

In [ ]:
results = []

for file in tqdm(sorted(edges_dir_old.glob("edges_*.parquet")), total=len(list(nodes_dir.glob("edges_*.parquet")))):
        cols = ["source", "target"]
        df_old = canon_edges(pd.read_parquet(file, columns=cols))
        df_new = canon_edges(pd.read_parquet(edges_dir_new / file.name, columns=cols))

        merged = df_old.merge(df_new, on=["source", "target"],
                              how="outer", indicator=True)

        n_communes = (merged["_merge"] == "both").sum()
        n_old_only = (merged["_merge"] == "left_only").sum()
        n_new_only = (merged["_merge"] == "right_only").sum()

        results.append({
            "dataset": "SYNTHETIC",
            "patient": patient_id,
            "sample": sample_id,
            "n_old": len(df_old),
            "n_new": len(df_new),
            "communes": n_communes,
            "seulement_old": n_old_only,
            "seulement_new": n_new_only,
            "jaccard": n_communes / len(merged) if len(merged) else 1.0,
        })

df_results_SYNT = pd.DataFrame(results)

204it [00:18, 10.82it/s]


### Assortativity

In [ ]:
if RUN_LONG:
    print("Computing z-scored assortativity")
    start = time()
    net_stats_SYNTHETIC = mosna.groups_assort_mixmat(
        net_dir=edges_dir, 
        attributes_col="phenotype", 
        make_onehot=True,
        id_level_1='patient',
        id_level_2=None, 
        extension='parquet',
        n_shuffle=n_shuffle,
        parallel_groups='max',  # or False
        save_intermediate_results=True)
    end = time()
    duration = end - start
    print(f"Assortativity took {duration:.3f}s for n_suffle {n_shuffle}")

    time_assort.append(['Synthetic', duration])

    net_stats_SYNTHETIC.index = net_stats_SYNTHETIC['id']
    net_stats_SYNTHETIC.drop(columns=['id'], inplace=True)

Computing z-scored assortativity


2026-11-11 16:50:01,319 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 38GB due to system memory limit of 31.04 GiB
2026-11-11 16:50:01,320 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 38GB due to system memory limit of 31.04 GiB
2026-11-11 16:50:01,322 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 38GB due to system memory limit of 31.04 GiB
2026-11-11 16:50:01,324 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 38GB due to system memory limit of 31.04 GiB
2026-11-11 16:50:01,325 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 38GB due to system memory limit of 31.04 GiB
2026-11-11 16:50:01,327 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 38GB due to system memory limit of 31.04 GiB
2026-11-11 16:50:01,329 - distributed.nanny.memory - WARNING - Ignoring provided memory limit 38GB due to system memory limit of 31.04 GiB
2026-11-11 16:50:01,331 - d

KeyboardInterrupt: 

In [ ]:
!mosna assortativity --file ./config_SYN.yaml --working_dir ../data/processed/SYNTHETIC/MOSNA_GUI --benchmark

[QT_INFO] [PROCESS] Compute Assortativity
[QT_PROGRESS] current=0 total=204 desc=[PROCESS] Compute Assortativity
[QT_PROGRESS] current=1 total=204 desc=[PROCESS] Compute Assortativity
[QT_PROGRESS] current=2 total=204 desc=[PROCESS] Compute Assortativity
[QT_PROGRESS] current=3 total=204 desc=[PROCESS] Compute Assortativity
[QT_PROGRESS] current=4 total=204 desc=[PROCESS] Compute Assortativity
[QT_PROGRESS] current=5 total=204 desc=[PROCESS] Compute Assortativity
[QT_PROGRESS] current=6 total=204 desc=[PROCESS] Compute Assortativity
[QT_PROGRESS] current=7 total=204 desc=[PROCESS] Compute Assortativity
[QT_PROGRESS] current=8 total=204 desc=[PROCESS] Compute Assortativity
[QT_PROGRESS] current=9 total=204 desc=[PROCESS] Compute Assortativity
[QT_PROGRESS] current=10 total=204 desc=[PROCESS] Compute Assortativity
[QT_PROGRESS] current=11 total=204 desc=[PROCESS] Compute Assortativity
[QT_PROGRESS] current=12 total=204 desc=[PROCESS] Compute Assortativity
[QT_PROGRESS] current=13 total=2

In [ ]:
tab = pd.DataFrame(tab, columns=["patient", "sample",'time', "number of cells", "dataset"])
tab.to_csv(f'../result/tysserand_py_times_total_preliminary.csv', index=False, mode='a', sep=',')

## Atera

### Benchmark

In [ ]:
output_dir = Path("../data/processed/Atera")
edges_dir = Path("../data/processed/Atera/tysserand_python_network")

nodes_dir = Path("../data/Atera/")
edges_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
if __name__ == "__main__":
    files = sorted(nodes_dir.glob("nodes*.parquet"))

    if RUN_LONG:
        worker = partial(
            process_file,
            pos_cols=[["x_centroid", "y_centroid"]],
            trim_dist=trim_dist,
            min_neighbors=min_neighbors,
            edges_dir=edges_dir,
        )

        n_workers = max(1, mp.cpu_count() - 1)
        with mp.Pool(processes=n_workers) as pool:
            tab = list(tqdm(pool.imap(worker, files, chunksize=1), total=len(files)))

        df = pd.DataFrame(
            tab, columns=["patient", "sample", "time", "number of cells", "dataset"]
        )
        df.to_csv(output_dir / "benchmark_times.csv", index=False)

NameError: name 'pos_cols' is not defined

In [ ]:
!mosna tysserand-network --file ./config_Atera.yaml --working_dir ../data/processed/Atera/MOSNA_GUI --benchmark

print(f"Average time per sample: {sum([x[2] for x in tab]) / len(tab):.2f} seconds")
print(f"Total time per sample: {sum([x[2] for x in tab])} seconds")

### Plotting Benchmark

In [ ]:
edges_dir_old = edges_dir
edges_dir_new = Path('../data/processed/Atera/MOSNA_GUI/temp/net_dir_mosna')

In [ ]:
results = []

for file in tqdm(sorted(edges_dir_old.glob("edges_*.parquet")), total=len(list(nodes_dir.glob("edges_*.parquet")))):
        cols = ["source", "target"]
        df_old = canon_edges(pd.read_parquet(file, columns=cols))
        df_new = canon_edges(pd.read_parquet(edges_dir_new / file.name, columns=cols))

        merged = df_old.merge(df_new, on=["source", "target"],
                              how="outer", indicator=True)

        n_communes = (merged["_merge"] == "both").sum()
        n_old_only = (merged["_merge"] == "left_only").sum()
        n_new_only = (merged["_merge"] == "right_only").sum()

        results.append({
            "dataset": "Atera",
            "patient": patient_id,
            "sample": sample_id,
            "n_old": len(df_old),
            "n_new": len(df_new),
            "communes": n_communes,
            "seulement_old": n_old_only,
            "seulement_new": n_new_only,
            "jaccard": n_communes / len(merged) if len(merged) else 1.0,
        })

df_results_Atera = pd.DataFrame(results)

### Assortativity

In [ ]:
if RUN_LONG:
    print("Computing z-scored assortativity")
    start = time()
    net_stats_SYNTHETIC = mosna.groups_assort_mixmat(
        net_dir=edges_dir, 
        attributes_col="group", 
        make_onehot=True,
        id_level_1='patient',
        id_level_2=None, 
        extension='parquet',
        n_shuffle=n_shuffle,
        parallel_groups='max',  # or False
        save_intermediate_results=True)
    end = time()
    duration = end - start
    print(f"Assortativity took {duration:.3f}s for n_suffle {n_shuffle}")

    time_assort.append(['Atera', duration])

    net_stats_SYNTHETIC.index = net_stats_SYNTHETIC['id']
    net_stats_SYNTHETIC.drop(columns=['id'], inplace=True)

In [ ]:
!mosna assortativity --file ./config_Atera.yaml --working_dir ../data/processed/Atera/MOSNA_GUI --benchmark

## Result

In [ ]:
df_results = pd.concat([df_results_CODEX, df_results_IMC, df_results_SYNT, df_results_Atera], ignore_index=True)
df_results.to_csv(f'../result/network_comparaison.csv', index=False, mode='a', sep=',')

,patient,sample,n_old,n_new,communes,seulement_old,seulement_new,jaccard
0,1,reg001,3472,3472,3472,0,0,1.0
1,1,reg002,6417,6417,6417,0,0,1.0
2,1,reg003,3027,3027,3027,0,0,1.0
3,1,reg004,1954,1954,1954,0,0,1.0
4,2,reg006,1561,1561,1561,0,0,1.0
...,...,...,...,...,...,...,...,...
1062,MB-0282,1,383499,383499,383499,0,0,1.0
1063,MB-0282,1,331741,331741,331741,0,0,1.0
1064,MB-0282,1,219020,219020,219020,0,0,1.0
1065,MB-0282,1,191622,191622,191622,0,0,1.0


In [ ]:
tab = pd.DataFrame(tab, columns=["patient", "sample",'time', "number of cells", "dataset"])
tab.to_csv(f'../result/tysserand_py_times_total.csv', index=False, mode='a', sep=',')

Concatenation des fichiers de bench

tysserand network

In [ ]:
import glob
files = glob.glob("../data/processed/**/MOSNA_GUI/benchmark/tysserand_*.csv", recursive=True)
dfs = []
for file in files:
    df = pd.read_csv(file)
    dfs.append(df)

benchmark_tysserand_Rust = pd.concat(dfs, ignore_index=True)
benchmark_tysserand_Rust.to_csv(f'../result/tysserand_Rust_times_total.csv', index=False, mode='a', sep=',')

assort files

In [ ]:
import glob
files = glob.glob("../data/processed/**/MOSNA_GUI/benchmark/assort_*.csv", recursive=True)
dfs = []
for file in files:
    df = pd.read_csv(file)
    dfs.append(df)

benchmark_assort_Rust = pd.concat(dfs, ignore_index=True)
benchmark_assort_Rust.to_csv(f'../result/Assortativity_Rust_times_total.csv', index=False, mode='a', sep=',')

time_assort_df = pd.DataFrame(time_assort, columns=["dataset", "time"])
time_assort_df.to_csv(f'../result/Assortativity_times_total_py.csv', index=False, mode='a', sep=',')

Assort file to save

In [ ]:
net_stats_CODEX.to_csv(f'../result/assort_py_CODEX.csv', index=False, mode='a', sep=',')
net_stats_IMC.to_csv(f'../result/assort_py_IMC.csv', index=False, mode='a', sep=',')
net_stats_SYNTHETIC.to_csv(f'../result/assort_py_SYNTHETIC.csv', index=False, mode='a', sep=',')

Plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

patient_col, sample_col = "patient", "sample"
n_cells_col = "number of cells"

py = tab[[patient_col, sample_col, n_cells_col, 'time']].rename(
    columns={'time': "time_python"})
rs = benchmark_tysserand_Rust[[patient_col, sample_col, 'Delaunay triangulation']].rename(
    columns={'Delaunay triangulation': "time_rust"})

df = py.merge(rs, on=[patient_col, sample_col], how="inner", validate="one_to_one")
df = df.sort_values(n_cells_col)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df[n_cells_col], df["time_python"], "o-", label="Python", markersize=4)
ax.plot(df[n_cells_col], df["time_rust"], "s-", label="Rust (Delaunay triangulation)", markersize=4)

ax.set_xlabel("Nombre de cellules")
ax.set_ylabel("Temps de calcul (s)")
ax.set_title("Temps de calcul : Python vs Rust")
ax.grid(alpha=0.3)
ax.legend()

# ax.set_xscale("log"); ax.set_yscale("log")

fig.tight_layout()
fig.savefig("../results/temps_python_vs_rust.png", dpi=600)
plt.show()